# TCC

## Importando Bibliotecas

In [ ]:
# Importando a biblioteca pandas
import pandas as pd

# Importando a biblioteca matplotlib
import matplotlib.pyplot as plt

# Importando a biblioteca seaborn
import seaborn as sns

# Importando a biblioteca numpy
import numpy as np

# Importando a kprototypes
from kmodes.kprototypes import KPrototypes

## Importando base de dados

In [ ]:
# Importando o arquivo csv
df_raw = pd.read_csv('cat_acidentes.csv', sep=';')

## Verificando a base

In [ ]:
# Utilizando o método describe() para visualizar as informações estatísticas do arquivo
df_raw.describe()

In [ ]:
# Utilizando o método info() para visualizar as informações do arquivo
df_raw.info()

In [ ]:
# Mostrar o data set em que tenha apenas valores nulos na coluna regiao
df_raw[df_raw['regiao'].isnull()]

## Data Wrangling

In [ ]:
# Achar outros valores em que log1 e a R DONA LAURA e log 2 e R MARIANTE
df_raw[(df_raw['log1'] == 'R DONA LAURA') & (df_raw['log2'] == 'R MARIANTE')]

# Mostrar a coluna regiao destes valores
df_raw[(df_raw['log1'] == 'R DONA LAURA') & (df_raw['log2'] == 'R MARIANTE')]['regiao']

In [ ]:
# Toda regiao em que o log1 for R DONA LAURA e o log2 for R MARIANTE sera igual a LESTE
df_raw.loc[(df_raw['log1'] == 'R DONA LAURA') & (df_raw['log2'] == 'R MARIANTE'), 'regiao'] = 'LESTE'

In [ ]:
# Excluir as linhas em que regiao for nulo
df_raw = df_raw.dropna(subset=['regiao'])

In [ ]:
# Contar colunas com nulos e mostrar apenas as que tem nulos
df_raw.isnull().sum()[df_raw.isnull().sum() > 0]

In [ ]:
# Excluir as colunas que contenham valores nulos
df_raw = df_raw.dropna(axis=1)

In [ ]:
# Contas colunas com nulos
df_raw.isnull().sum()

In [ ]:
# Tornar o idacidente a coluna de index
df_raw = df_raw.set_index('idacidente')

In [ ]:
# Excluir colunas data_extracao, queda_arr e idacidente
df_raw = df_raw.drop(['data_extracao', 'queda_arr'], axis=1)

In [ ]:
# Excluir os numeros apos o espaco na coluna data
df_raw['data'] = df_raw['data'].str.split(' ').str[0]

In [ ]:
# Sabendo-se que o segundo numero apos o - e o mes, criar uma coluna chamada mes com esses valores
df_raw['mes'] = df_raw['data'].str.split('-').str[1]

In [ ]:
# Trocar o numero do mes por seu nome
df_raw['mes'] = df_raw['mes'].replace({'01': 'JANEIRO',
                                       '02': 'FEVEREIRO',
                                       '03': 'MARÇO',
                                       '04': 'ABRIL',
                                       '05': 'MAIO',
                                       '06': 'JUNHO',
                                       '07': 'JULHO',
                                       '08': 'AGOSTO',
                                       '09': 'SETEMBRO',
                                       '10': 'OUTUBRO',
                                       '11': 'NOVEMBRO',
                                       '12': 'DEZEMBRO'})

In [ ]:
df_raw['mes'].head()

In [ ]:
# Sabendo-se que o primeiro numero antes do - e o ano, criar uma coluna chamada ano com esses valores
df_raw['ano'] = df_raw['data'].str.split('-').str[0]

In [ ]:
# Excluir a coluna data
df_raw = df_raw.drop('data', axis=1)

In [ ]:
# Mostrar apenas as colunas que contem nulos
df_raw.isnull().sum()[df_raw.isnull().sum() > 0]

In [ ]:
# Transformar as colunas tipo_acid, dia_sem, noite_dia, regiao, mes e ano em category
df_raw['tipo_acid'] = df_raw['tipo_acid'].astype('category')
df_raw['dia_sem'] = df_raw['dia_sem'].astype('category')
df_raw['noite_dia'] = df_raw['noite_dia'].astype('category')
df_raw['regiao'] = df_raw['regiao'].astype('category')
df_raw['mes'] = df_raw['mes'].astype('category')
df_raw['ano'] = df_raw['ano'].astype('category')


In [ ]:
# Criar uma lista com os nomes das colunas categoricas, que sao do tipo category
colunas_categoricas = df_raw.select_dtypes(include=['category']).columns

In [ ]:
# Create a copy for the standardized data
df_std = df_raw.copy()

# Standarize all numerical columns with a for loop
for col in df_std.select_dtypes(include=['int64', 'float64']).columns:
    df_std[col] = (df_std[col] - df_std[col].mean()) / df_std[col].std()


In [ ]:
# Criar um dataset com as colunas numericas usando np.number
data_numerical = df_std.select_dtypes(include=[np.number])

In [ ]:
# Creating a list the had the index of the categorical columns
categorical_columns = [df_std.columns.get_loc(col) for col in colunas_categoricas]

In [ ]:
categorical_columns

## Analise da distribuicao

In [ ]:
# Importing silhouette_score, calinski_harabasz_score and davies_bouldin_score
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
# Creating a list that will receive the number of clusters from 2 to 10
k_values = range(2, 11)

# For loop that will validate the number of clusters wish silhouette score, calinski_harabasz and davies_bouldin
for k in k_values:
    # Creating the model
    kproto = KPrototypes(n_clusters=k, init='Huang', verbose=2, n_init=20, n_jobs=-1)
    # Fitting the model
    cluster_assignments = kproto.fit_predict(df_raw, categorical=categorical_columns)
    # Calculating the silhouette score
    silhouette = silhouette_score(data_numerical, cluster_assignments)
    # Calculating the calinski_harabasz score
    calinski_harabasz = calinski_harabasz_score(data_numerical, cluster_assignments)
    # Calculating the davies_bouldin score
    davies_bouldin = davies_bouldin_score(data_numerical, cluster_assignments)
    # Printing the results
    print('Silhouette score for k = {}: {}'.format(k, silhouette))
    print('Calinski Harabasz score for k = {}: {}'.format(k, calinski_harabasz))
    print('Davies Bouldin score for k = {}: {}'.format(k, davies_bouldin))
    print('')

## Com a coluna tipo_acid

In [ ]:
# Creating a copy for the raw data
df = df_raw.copy()

# Creating the categorical columns index for the new dataframe
categorical_columns = [df.columns.get_loc(col) for col in colunas_categoricas]

# Creating the model
kproto2 = KPrototypes(n_clusters=3, init='Huang', verbose=2, n_init=20, n_jobs=-1)

# Fitting the model
cluster_assignments2 = kproto2.fit_predict(df_raw, categorical=categorical_columns)

# Creating a new column with the cluster assignments in  new df
df['cluster'] = cluster_assignments2

In [ ]:
# Analysing the distribution of the clusters
df['cluster'].value_counts()

In [ ]:
# Analyzing clusters with the mean of the numerical columns
df.groupby('cluster').mean()

In [ ]:
# Analyze clusters with the mean of the categorical columns
df.groupby('cluster').agg(lambda x: x.value_counts().index[0])

In [ ]:
# Split the raw dataframe in three dataframes, one for each cluster
df_cluster_0 = df[df['cluster'] == 0]
df_cluster_1 = df[df['cluster'] == 1]
df_cluster_2 = df[df['cluster'] == 2]

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 0
fig, axes = plt.subplots(2, 3, figsize=(40, 20))


# Set a title
fig.suptitle('Distribuição de acidentes por cluster (Cluster 0)', fontsize=30)

sns.countplot(x='dia_sem', data=df_cluster_0, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_0, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_0, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_0, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_0, ax=axes[1, 1])
sns.countplot(x='tipo_acid', data=df_cluster_0, ax=axes[1, 2])

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 1
fig, axes = plt.subplots(2, 3, figsize=(40, 20))


# Set a title
fig.suptitle('Distribuição de acidentes por cluster (Cluster 1)', fontsize=30)

sns.countplot(x='dia_sem', data=df_cluster_1, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_1, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_1, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_1, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_1, ax=axes[1, 1])
sns.countplot(x='tipo_acid', data=df_cluster_1, ax=axes[1, 2])

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 2
fig, axes = plt.subplots(2, 3, figsize=(40, 20))

# Set a title
fig.suptitle('Distribuição de acidentes por cluster (Cluster 2)', fontsize=30)

sns.countplot(x='dia_sem', data=df_cluster_2, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_2, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_2, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_2, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_2, ax=axes[1, 1])
sns.countplot(x='tipo_acid', data=df_cluster_2, ax=axes[1, 2])

## Retirando a coluna tipo_acid e clustering


In [ ]:
df_std_acid = df_std.drop('tipo_acid', axis=1)

In [ ]:
# Creating a list the had the index of the categorical columns
categorical_columns = [df_std_acid.columns.get_loc(col) for col in df_std_acid.select_dtypes(include=['category']).columns]

In [ ]:
# Creating a model with three clusters
kproto = KPrototypes(n_clusters=3, init='Huang', verbose=2, n_init=20, n_jobs=-1)

# Fitting the model
cluster_assignments = kproto.fit_predict(df_std_acid, categorical=categorical_columns)

In [ ]:
silhouette = silhouette_score(data_numerical, cluster_assignments)
calinski_harabasz = calinski_harabasz_score(data_numerical, cluster_assignments)
davies_bouldin = davies_bouldin_score(data_numerical, cluster_assignments)

print(silhouette)
print(calinski_harabasz)
print(davies_bouldin)

In [ ]:
# Create a copy of df
df_acid = df_raw.copy()

In [ ]:
# Creating a list with the cluster assignments
cluster_assignments = list(cluster_assignments)

# Creating a new column with the cluster assignments
df_acid['cluster'] = cluster_assignments

In [ ]:
# Analysing the distribution of the clusters
df_acid['cluster'].value_counts()

In [ ]:
# Analyzing clusters with the mean of the numerical columns
df_acid.groupby('cluster').mean()

In [ ]:
# Split the raw dataframe in three dataframes, one for each cluster
df_cluster_0 = df_acid[df_acid['cluster'] == 0]
df_cluster_1 = df_acid[df_acid['cluster'] == 1]
df_cluster_2 = df_acid[df_acid['cluster'] == 2]

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 0
fig, axes = plt.subplots(2, 3, figsize=(40, 20))
sns.countplot(x='dia_sem', data=df_cluster_0, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_0, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_0, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_0, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_0, ax=axes[1, 1])

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 1
fig, axes = plt.subplots(2, 3, figsize=(40, 20))
sns.countplot(x='dia_sem', data=df_cluster_1, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_1, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_1, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_1, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_1, ax=axes[1, 1])

In [ ]:
# Analyze the distribution with matplot lib and seaborn for each categorical data in the cluster 2
fig, axes = plt.subplots(2, 3, figsize=(40, 20))
sns.countplot(x='dia_sem', data=df_cluster_2, ax=axes[0, 0])
sns.countplot(x='noite_dia', data=df_cluster_2, ax=axes[0, 1])
sns.countplot(x='regiao', data=df_cluster_2, ax=axes[0, 2])
sns.countplot(x='mes', data=df_cluster_2, ax=axes[1, 0])
sns.countplot(x='ano', data=df_cluster_2, ax=axes[1, 1])

In [ ]:
# Exportar o df_raw para csv
df_raw.to_csv('df_raw.csv', index=False)